# 03 — Couche silver : nettoyage et minimisation des données
Étape 1 : inspecter les formats réels des fichiers bruts avant d'écrire le nettoyage.


In [ ]:
import sys
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)

PROJET = Path("/mnt/c/Users/Salah/Desktop/SecuMon-IA")
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists()), PROJET)
RAW = ROOT / "data" / "raw"
BRONZE = ROOT / "data" / "processed" / "bronze" / "transactions.parquet"

print("Python        :", sys.executable)
print("Dossier       :", Path.cwd())
print("Racine        :", ROOT)
print("RAW existe    :", RAW.exists())
print("BRONZE existe :", BRONZE.exists())

con = duckdb.connect()

In [ ]:
fichiers = sorted(RAW.rglob("*.csv"))
pd.DataFrame({
    "fichier":   [str(f.relative_to(ROOT)) for f in fichiers],
    "taille_Mo": [round(f.stat().st_size / 1e6, 1) for f in fichiers],
})

In [ ]:
def trouver(nom):
    res = [f for f in fichiers if f.name == nom]
    if not res:
        raise FileNotFoundError(f"{nom} introuvable sous {RAW} -> envoie-moi le resultat de la cellule 3")
    return res[0]

USERS = trouver("sd254_users.csv")
CARDS = trouver("sd254_cards.csv")

for p in (USERS, CARDS):
    n = con.sql(f"SELECT COUNT(*) AS n FROM read_csv('{p}', all_varchar=true)").df()["n"].iloc[0]
    print(f"{p.name} : {n} lignes")
    cols = con.sql(f"DESCRIBE SELECT * FROM read_csv('{p}', all_varchar=true)").df()["column_name"]
    print(list(cols))
    print()

In [ ]:
con.sql(rf"""
    SELECT
        COUNT(*)                                                                      AS lignes,
        COUNT(*) FILTER (WHERE "Time" IS NULL
                            OR NOT regexp_full_match("Time", '\d{{2}}:\d{{2}}'))      AS time_mal_forme,
        COUNT(*) FILTER (WHERE "Amount" IS NULL
                            OR NOT regexp_full_match("Amount", '-?\$-?\d+\.\d{{2}}')) AS amount_mal_forme,
        COUNT(*) FILTER (WHERE "Zip" IS NULL OR "Zip" = '')                           AS zip_vide,
        COUNT(*) FILTER (WHERE "Merchant State" IS NULL OR "Merchant State" = '')     AS state_vide,
        MIN("Amount") FILTER (WHERE "Amount" LIKE '%-%')                              AS exemple_negatif
    FROM '{BRONZE}'
""").df()

In [ ]:
for p in (USERS, CARDS):
    cols = con.sql(f"DESCRIBE SELECT * FROM read_csv('{p}', all_varchar=true)").df()["column_name"]
    print(f"=== {p.name} ({len(cols)} colonnes) ===")
    for i, c in enumerate(cols):
        print(f"{i:2d}  {c}")
    print()

## Formats des fichiers clients et cartes
Pour chaque colonne, on affiche un **motif** au lieu de la valeur : chiffres → `9`, lettres → `A`.
Exemple : `$59696` → `$99999`. On voit ainsi le format sans afficher aucune donnée (numéro de carte, CVV, adresse).

In [ ]:
import re

pd.set_option("display.max_colwidth", None)

users = pd.read_csv(USERS, dtype=str)
cards = pd.read_csv(CARDS, dtype=str)

SENSIBLES = {"Person", "Address", "Apartment", "Card Number", "CVV"}

def motif(v):
    if pd.isna(v):
        return "<vide>"
    return re.sub(r"[A-Za-z]", "A", re.sub(r"\d", "9", v))

def resume(df):
    lignes = []
    for c in df.columns:
        s = df[c]
        motifs = s.map(motif).value_counts()
        valeurs = ""
        if c not in SENSIBLES and s.nunique() <= 6:
            valeurs = ", ".join(sorted(s.dropna().unique()))
        lignes.append({
            "colonne": c,
            "vides": int(s.isna().sum()),
            "distinctes": s.nunique(),
            "motifs_principaux": ", ".join(f"{m} ({n})" for m, n in motifs.head(3).items()),
            "valeurs (si peu)": valeurs,
        })
    return pd.DataFrame(lignes)

ATTENDU_CARTES = ["User", "CARD INDEX", "Card Brand", "Card Type", "Card Number", "Expires",
                  "CVV", "Has Chip", "Cards Issued", "Credit Limit", "Acct Open Date",
                  "Year PIN last Changed", "Card on Dark Web"]
print("Colonnes cartes manquantes :", [c for c in ATTENDU_CARTES if c not in cards.columns] or "aucune")
print("Colonnes cartes en plus    :", [c for c in cards.columns if c not in ATTENDU_CARTES] or "aucune")

print("\nClients :", len(users), "lignes")
print("Identifiants client dans les cartes : min =", cards["User"].astype(int).min(),
      "max =", cards["User"].astype(int).max())

print("\n=== CLIENTS ===")
display(resume(users))
print("=== CARTES ===")
display(resume(cards))

## Construction de la couche silver
Le code est dans `secumon/ml/data/ibm.py` (testé par `tests/test_ibm_silver.py`) ;
le notebook l'appelle et contrôle le résultat.

In [ ]:
import importlib
import time

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from secumon.ml.data import ibm
importlib.reload(ibm)

SILVER = ROOT / "data" / "processed" / "silver"

t0 = time.perf_counter()
comptes = ibm.build_silver(RAW, BRONZE, SILVER)
print(comptes, f"-> {time.perf_counter() - t0:.0f} s")

for f in sorted(SILVER.glob("*.parquet")):
    print(f"{f.name:28s} {f.stat().st_size / 1e6:8.1f} Mo")

In [ ]:
TX = SILVER / "transactions.parquet"
CARDS_S = SILVER / "cards.parquet"
CUST_S = SILVER / "customers.parquet"

controles = con.sql(f"""
    SELECT COUNT(*)                                                  AS transactions,
           COUNT(*) FILTER (WHERE ts IS NULL OR amount IS NULL
                               OR channel IS NULL OR card_id IS NULL) AS valeurs_manquantes,
           COUNT(*) FILTER (WHERE is_refund)                          AS remboursements,
           COUNT(*) FILTER (WHERE is_online AND channel <> 'online')  AS online_incoherent,
           MIN(ts)                                                    AS premiere,
           MAX(ts)                                                    AS derniere
    FROM '{TX}'
""").df()

cartes_inconnues = con.sql(f"""
    SELECT COUNT(*) AS n FROM '{TX}' t ANTI JOIN '{CARDS_S}' c USING (card_id)
""").df()["n"].iloc[0]

clients_inconnus = con.sql(f"""
    SELECT COUNT(*) AS n FROM '{TX}' t ANTI JOIN '{CUST_S}' c USING (user_id)
""").df()["n"].iloc[0]

display(controles)
print("Transactions dont la carte est inconnue  :", cartes_inconnues)
print("Transactions dont le client est inconnu  :", clients_inconnus)

In [ ]:
con.sql(f"""
    SELECT split,
           COUNT(*)                     AS transactions,
           SUM(is_fraud::INT)::BIGINT   AS fraudes
    FROM '{TX}'
    GROUP BY split
    ORDER BY split
""").df()

In [ ]:
print("Colonnes clients :", pd.read_parquet(CUST_S).columns.tolist())
print("Colonnes cartes  :", pd.read_parquet(CARDS_S).columns.tolist())
display(con.sql(f"SELECT * FROM '{TX}' LIMIT 5").df())

## Conclusion
Couche silver construite en 16 s par `secumon/ml/data/ibm.py` (9 tests unitaires) :

| Fichier | Lignes | Contenu |
|---|---|---|
| customers.parquet | 2 000 | profil financier et géographique, sans nom/adresse/genre |
| cards.parquet | 6 146 | type, plafond, puce, dates — sans numéro de carte ni CVV |
| transactions.parquet | 24 386 900 | horodatage, montant, remboursement, canal, commerçant, split |

Contrôles : 0 valeur manquante, 0 transaction orpheline, découpage identique au notebook 02.
Les 7 601 transactions « puce + ONLINE » sont conservées et repérables (`is_online AND channel <> 'online'`).